In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    batch_size_train = 1024
    batch_size_infer = 1024
    n_epochs = 10
    max_len = 10
    model_name = "last_week_try"
    hidden_dim = 32
    embedding_dim = 64


config = Config()

In [4]:
print(utils.recall_vals_one_row([1, 2, 3], 2))
print(utils.recall_vals_one_row([1, 2, 3], 4))

True
False


In [5]:
utils.recall_several_targets(candidates=[[1, 2, 3], [1]], targets=[[1, 3, 2], [1, 2]])

0.8

In [6]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=16, progress_bar=True, use_memory_fs=False)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [7]:
%%time

PATH_DATA = '../data/02_intermediate/'
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
df_train = df_train.to_pandas()
df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

df_train.name = "df_train"
df_test.name = "df_test"

df_train = utils.make_date_stats(df_train)
df_test = utils.make_date_stats(df_test)

df_train.drop(['date_day'], axis=1, inplace=True)
df_test.drop(['date_day'], axis=1, inplace=True)

df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

CPU times: user 1min 55s, sys: 25.4 s, total: 2min 20s
Wall time: 2min 6s


In [8]:
df_train.select(pl.col("n_week").unique())

n_week
i8
31
32
33
34


In [9]:
df_train = df_train.filter(pl.col("n_week") == 34)

In [10]:
%%time

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

CPU times: user 1min 42s, sys: 12 s, total: 1min 54s
Wall time: 18.3 s


In [11]:
del df_train, df_test
gc.collect()

2771

In [12]:
# ### IF DEBUG PIPELINE !!!

# df = df.filter(pl.col("user") % 100 == 0)

In [13]:
counts_aid = df.groupby([config.col_aid]).count().sort("count", reverse=True)

In [14]:
df.head()

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,num_action
i64,i64,i64,str,i32,datetime[ns],i8,i8,i8,i8,i32
14571581,1100210,1662328791563,"""clicks""",0,2022-09-04 23:59:51.563000064,4,35,6,23,0
14571515,781377,1662328789887,"""clicks""",0,2022-09-04 23:59:49.887000064,4,35,6,23,0
14571170,336732,1662328788088,"""clicks""",0,2022-09-04 23:59:48.088,4,35,6,23,0
14568399,104775,1662328781349,"""clicks""",0,2022-09-04 23:59:41.348999936,4,35,6,23,0
14571580,202353,1662328781067,"""clicks""",0,2022-09-04 23:59:41.067000064,4,35,6,23,0


In [15]:
counts_aid.head()

aid,count
i64,u32
485256,135892
1460571,46096
108125,36759
1551213,35912
1406660,33238


In [16]:
df.shape

(59689038, 11)

In [17]:
df = df.to_pandas()

In [18]:
def split_data(df, validation_cut, test_users):
    df_val = df[
        (df[config.col_num_action] <= validation_cut)
        & (df[config.col_user].isin(test_users))
    ]

    df_val_seq = df[
        (df[config.col_num_action] > validation_cut)
        & (df[config.col_user].isin(test_users))
    ]

    df_train = df.drop(df_val.index)
    return df_train, df_val, df_val_seq

In [19]:
%%time

X_train, X_val, X_val_seq = split_data(df, validation_cut=2, test_users=test_users)

CPU times: user 15.6 s, sys: 1.58 s, total: 17.1 s
Wall time: 17.2 s


In [20]:
%%time

trs = 20
counts_aid_filtered = counts_aid.filter(pl.col("count") >= trs).to_pandas()
pad = len(counts_aid_filtered)

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=pad)
ordinal_encoder.fit(counts_aid_filtered[config.col_aid].values.reshape(-1, 1))

CPU times: user 255 ms, sys: 7.75 ms, total: 263 ms
Wall time: 260 ms


OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=395762)

In [21]:
%%time

X_train[config.col_aid] = ordinal_encoder.transform(
    X_train[config.col_aid].values.reshape(-1, 1)
).astype(int)
X_val_seq[config.col_aid] = ordinal_encoder.transform(
    X_val_seq[config.col_aid].values.reshape(-1, 1)
).astype(int)

CPU times: user 21.6 s, sys: 754 ms, total: 22.3 s
Wall time: 22.4 s


In [22]:
print(X_train.shape)
X_train = X_train[X_train[config.col_aid] != pad]
print(X_train.shape)

(56468270, 11)
(49524224, 11)


In [23]:
%%time

X_train = pl.from_pandas(X_train)
X_train = X_train.sort(config.col_ts)

X_val = pl.from_pandas(X_val)
X_val = X_val.sort(config.col_ts)

X_val_seq = pl.from_pandas(X_val_seq)
X_val_seq = X_val_seq.sort(config.col_ts)

CPU times: user 37.9 s, sys: 12.3 s, total: 50.2 s
Wall time: 4.97 s


In [24]:
X_train.filter(pl.col(config.col_user) == 0)

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,num_action
i64,i64,i64,str,i32,datetime[ns],i8,i8,i8,i8,i32
0,2225,1661157951797,"""clicks""",32,2022-08-22 10:45:51.796999936,22,34,0,10,128
0,46907,1661157962188,"""clicks""",32,2022-08-22 10:46:02.188,22,34,0,10,127
0,243014,1661157993281,"""clicks""",32,2022-08-22 10:46:33.280999936,22,34,0,10,126
0,194953,1661158034888,"""clicks""",32,2022-08-22 10:47:14.888,22,34,0,10,125
0,102261,1661158090652,"""clicks""",32,2022-08-22 10:48:10.652,22,34,0,10,124
0,244048,1661158184827,"""clicks""",32,2022-08-22 10:49:44.827000064,22,34,0,10,123
0,102261,1661158189101,"""clicks""",32,2022-08-22 10:49:49.100999936,22,34,0,10,122
0,109933,1661158253105,"""clicks""",32,2022-08-22 10:50:53.104999936,22,34,0,10,121
0,307928,1661206127572,"""clicks""",33,2022-08-23 00:08:47.572,23,34,1,0,119


In [25]:
%%time

X_train = (
    X_train.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
)

X_val = (
    X_val.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

X_val_seq = (
    X_val_seq.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

CPU times: user 1min 3s, sys: 22.8 s, total: 1min 26s
Wall time: 21.9 s


In [26]:
X_train.shape

(4399896, 2)

In [27]:
X_train.filter(pl.col(config.col_user) == 0)

user,aid
i64,list[i64]
0,"[2225, 46907, ... 34629]"


In [28]:
# dict_map_targets = dict()
# dict_map_targets["clicks"] = 0
# dict_map_targets["carts"] = 1
# dict_map_targets["orders"] = 2

In [29]:
counts_aid["aid"].min(), counts_aid["aid"].max()

(0, 1855602)

In [30]:
pad

395762

In [31]:
# %%time

# df_train = df_train.to_pandas()
# df_train['aid_filtered'] = ordinal_encoder.transform(df_train['aid'].values.reshape(-1, 1)).astype(int)
# df_train = pl.from_pandas(df_train)

In [32]:
# counts_aid_type.filter((pl.col("type") == "orders") & (pl.col("count") >= 1)).select(
#     ["aid"]
# ).unique().shape

In [33]:
# counts_aid_type

In [34]:
counts_aid.tail()

aid,count
i64,u32
945025,1
224129,1
1076097,1
551809,1
1764225,1


In [35]:
counts_aid.filter(pl.col("aid") == 1)

aid,count
i64,u32
1,2


In [36]:
# len(counts_aid), len(counts_aid_filtered)

In [37]:
# %%time

# token_to_idx = dict()
# # token_to_idx[-1] = 0 # pad
# token_to_idx[0] = 0 # unknown / pad

# idx = 1
# for i in tqdm(range(len(counts_aid["aid"]))):
#     aid = counts_aid["aid"][i]
#     cnt = counts_aid["count"][i]
#     if cnt > 4:
#         token_to_idx[aid] = idx
#         idx += 1
#     # else:
#     #     token_to_idx[aid] = 0

# idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [38]:
# len(token_to_idx)

In [39]:
# min(list(token_to_idx.values()))

In [40]:
# token_to_idx[0]

In [41]:
# max_len = 10
# # pad = counts_aid["aid"].max() + 1

# # def get_id_of_token(token):
# #     if token in set(list(token_to_idx.keys())[1:]):
# #         idx = token_to_idx[token]
# #     else:
# #         idx = 0  # unknown
# #     return idx


# # def get_token_if_id(idx):
# #     if idx in set(list(idx_to_token.keys())[1:]):
# #         token = idx_to_token[idx]
# #     else:
# #         token = 0  # unknown
# #     return token


# def to_matrix(names, max_len=max_len, pad=pad, dtype="int32", batch_first=True):
#     """Casts a list of names into rnn-digestible matrix"""

#     max_len = max_len or max(map(len, names))
#     names_ix = np.zeros([len(names), max_len], dtype) + pad

#     for i in tqdm(range(len(names))):
#         cur_name = names[i]
#         line_ix = cur_name[-max_len:]  # last max_len elements
#         # cur_name = cur_name[-max_len:]  # last max_len elements
#         # line_ix = [
#         #     get_id_of_token(cur_name[c]) for c in range(len(cur_name)) if c < max_len
#         # ]
#         names_ix[i, -len(line_ix) :] = line_ix

#     if not batch_first:  # convert [batch, time] into [time, batch]
#         names_ix = np.transpose(names_ix)

#     return names_ix

In [42]:
l = [1, 2, 3]
print(utils.split_list_randomly(l=l, seed=1))
print(utils.split_list_randomly(l=l, seed=2))

([1, 2], [3])
([1], [2, 3])


In [43]:
X_train.head()

user,aid
i64,list[i64]
5108608,"[112608, 117228, 81314]"
6027072,"[154119, 336846, ... 336846]"
12887808,"[150921, 153819, ... 153819]"
13832512,"[210219, 376958, ... 241542]"
14031360,"[369611, 280935, ... 136328]"


In [44]:
X_train.shape

(4399896, 2)

In [45]:
X_val.head()

user,aid
i64,list[i64]
13953536,[1146836]
13896832,[440558]
13127808,[995916]
13897152,"[668012, 337093]"
13974464,[331201]


In [46]:
%%time

X_train = X_train.to_pandas()
X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: x[:-1])
X_train[config.col_target] = X_train[config.col_aid].parallel_apply(lambda x: x[-1])
X_train = pl.from_pandas(X_train)

CPU times: user 43.3 s, sys: 7.49 s, total: 50.8 s
Wall time: 52.8 s


In [47]:
X_train.head()

user,aid,train_seq,target
i64,list[i64],list[i64],i64
5108608,"[112608, 117228, 81314]","[112608, 117228]",81314
6027072,"[154119, 336846, ... 336846]","[154119, 336846, ... 1689]",336846
12887808,"[150921, 153819, ... 153819]","[150921, 153819, ... 150921]",153819
13832512,"[210219, 376958, ... 241542]","[210219, 376958, ... 241542]",241542
14031360,"[369611, 280935, ... 136328]","[369611, 280935, ... 319471]",136328


In [48]:
# %%time

# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[:-1]).alias(config.col_train_seq)))
# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[-1]).alias(config.col_target)))

In [49]:
# %%time

# X_train['val'] = X_train["val"].progress_apply(lambda x: x[0]) #orig_vals

# col_aids = 'train'
# X_train = pl.from_pandas(X_train)
# X_train = (
#     X_train.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_train = X_train[['user', 'train', 'val']]

In [50]:
X_train.shape

(4399896, 4)

In [51]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
357888,17
324224,4
225088,5
135744,4
6272,26
212480,5
203776,5
345856,3
352192,16


In [52]:
%%time

X_train = X_train.to_pandas()

list_train = list()
list_target = list()
list_user = list()

n_iter = 15
for i in tqdm(range(len(X_train))):
    row = X_train.iloc[i]
    row_train = list(row[config.col_train_seq])
    target_cur_main = row[config.col_target]
    len_row_train = len(row_train)
    if len_row_train == 1 and target_cur_main < pad:
        list_train.append(tuple(row_train))
        list_target.append(target_cur_main)
        list_user.append(row[config.col_user])
    else:
        n_iter_cur = min(n_iter, len_row_train - 1)
        # train_cur_main = utils.truncate_list(l=row_train, max_len=config.max_len, from_start=False)
        if target_cur_main < pad:
            list_train.append(tuple(row_train))
            list_target.append(target_cur_main)
            list_user.append(row[config.col_user])

        for seed in range(n_iter_cur):
            train_cur, target_cur = utils.split_list_randomly(l=row_train, seed=seed)
            target_cur = target_cur[0]
            if target_cur < pad:
                # train_cur = utils.truncate_list(l=train_cur, max_len=config.max_len, from_start=False)
                list_train.append(tuple(train_cur))
                list_target.append(target_cur)
                list_user.append(row[config.col_user])

X_train = pd.DataFrame()
X_train[config.col_user] = list_user
X_train[config.col_train_seq] = list_train
X_train[config.col_target] = list_target
X_train.drop_duplicates(inplace=True)
X_train.reset_index(drop=True, inplace=True)

X_train = pl.from_pandas(X_train)

  0%|          | 0/4399896 [00:00<?, ?it/s]

CPU times: user 15min 44s, sys: 15.1 s, total: 15min 59s
Wall time: 15min 57s


In [53]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
5108608,"[112608, 117228]",81314
5108608,[112608],117228
6027072,"[154119, 336846, ... 1689]",336846
6027072,[154119],336846
6027072,"[154119, 336846]",193537


In [54]:
X_train.shape

(22591260, 3)

In [55]:
%%time

X_train = X_train.to_pandas()
X_train[config.col_train_seq] = X_train[config.col_train_seq].parallel_apply(lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False))
X_train = pl.from_pandas(X_train)

CPU times: user 5min 3s, sys: 35.4 s, total: 5min 39s
Wall time: 5min 37s


In [56]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
5108608,"[112608, 117228]",81314
5108608,[112608],117228
6027072,"[154119, 336846, ... 1689]",336846
6027072,[154119],336846
6027072,"[154119, 336846]",193537


In [57]:
X_train = X_train.filter(pl.col(config.col_target) != pad)

In [58]:
X_train[config.col_target].max(), pad  # (31207, 31211)

(395761, 395762)

In [59]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
276352,26
376128,13
383744,13
282752,16
80512,10
304896,30
28672,14
267904,14
363072,57


In [60]:
X_val_seq.head()

user,aid
i64,list[i64]
13460992,[126585]
13654144,"[176041, 312991, ... 312991]"
13003712,[20790]
13218240,"[98424, 98424, 98424]"
13831424,"[70787, 59259, ... 220845]"


In [61]:
X_val_seq = X_val_seq.to_pandas()
X_val_seq[config.col_train_seq] = X_val_seq[config.col_aid].parallel_apply(
    lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
)
X_val_seq = pl.from_pandas(X_val_seq)

In [62]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13460992,[126585],[126585]
13654144,"[176041, 312991, ... 312991]","[176041, 312991, ... 312991]"
13003712,[20790],[20790]
13218240,"[98424, 98424, 98424]","[98424, 98424, 98424]"
13831424,"[70787, 59259, ... 220845]","[70787, 59259, ... 220845]"


In [63]:
X_val = X_val.join(
    X_val_seq[[config.col_user, config.col_train_seq]], how="inner", on=config.col_user
)

In [64]:
X_val = X_val.rename({config.col_aid: config.col_target})

In [65]:
X_val.filter(pl.col(config.col_user) == 13833088)

user,target,train_seq
i64,list[i64],list[i64]
13833088,"[998481, 998481, 226025]","[108300, 124293, ... 212667]"


In [66]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
5108608,"[112608, 117228]",81314
5108608,[112608],117228
6027072,"[154119, 336846, ... 1689]",336846
6027072,[154119],336846
6027072,"[154119, 336846]",193537


In [67]:
X_train.shape

(22591260, 3)

In [68]:
X_train.shape

(22591260, 3)

In [69]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
146048,18
260800,19
40064,125
382464,89
242880,17
220928,11
19712,15
18752,130
158144,32


In [70]:
# %%time

# X_val["train"] = X_val["train"].progress_apply(lambda x: utils.truncate_list(l=list(x), max_len=max_len, from_start=False))
# X_val["val"] = X_val["val"].progress_apply(lambda x: x[0]) #orig_vals

In [71]:
# %%time

# X_val = X_val.explode('train')
# X_val['train'] = ordinal_encoder.transform(X_val['train'].values.reshape(-1, 1)).astype(int)

# col_aids = 'train'
# X_val = pl.from_pandas(X_val)
# X_val = (
#     X_val.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_val = X_val[['user', 'train', 'val']]

In [72]:
# %%time

# df_train_clicks["val"] = df_train_clicks["val"].progress_apply(lambda x: x[0])

In [73]:
# df_train_clicks["val"].value_counts()

In [74]:
# df_train_clicks.head()

In [75]:
pad

395762

In [76]:
# df_train[col_aids].value_counts().sort("counts", reverse=True)

In [77]:
# df_train_clicks.drop([col_aids, "split"], axis=1, inplace=True)

In [78]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [pl.col("aid").apply(lambda x: utils.split_list_randomly(list(x))).alias("split")]
# )

In [79]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [
#         pl.col("split").apply(lambda x: x[0]).alias("train"),
#         pl.col("split").apply(lambda x: x[1]).alias("val"),
#     ]
# )

# for col in ["aid", "split"]:
#     df_train_clicks.drop_in_place(col)

# df_train_clicks = df_train_clicks.sort("user")

In [80]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.to_pandas()

In [81]:
# %%time

# df_train_clicks_mapped["val"] = df_train_clicks_mapped["val"].parallel_apply(
#     lambda x: get_id_of_token(x[0])
# )

In [82]:
# shape_orig = df_train_clicks_mapped.shape
# df_train_clicks_mapped = df_train_clicks_mapped[
#     df_train_clicks_mapped["val"] != 0
# ].reset_index(drop=True)
# print(shape_orig, "->", df_train_clicks_mapped.shape)

In [83]:
# (12773, 4)

In [84]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
5108608,"[112608, 117228]",81314
5108608,[112608],117228
6027072,"[154119, 336846, ... 1689]",336846
6027072,[154119],336846
6027072,"[154119, 336846]",193537


In [85]:
X_val.head()

user,target,train_seq
i64,list[i64],list[i64]
13340928,"[1713158, 1421960, 1713158]","[213574, 276287, ... 270835]"
13570752,"[1035803, 1775391, 1641216]","[326363, 326363, ... 220643]"
13151872,"[660635, 852900, 1517855]","[339541, 238694]"
13704192,"[188965, 1034864, 188965]","[40288, 370788, ... 86709]"
14419456,"[827115, 313825, 51070]",[243984]


In [86]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].progress_apply(lambda x: utils.truncate_list(list(x), max_len=max_len))

In [87]:
# %%time

# df_train_clicks["train"] = list(to_matrix(df_train_clicks["train"]))

In [88]:
# df_train_clicks["val"].value_counts()

In [89]:
# df_train_clicks.head()

In [90]:
# df_train_clicks["train"][0]

In [91]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].parallel_apply(
#     lambda x: to_matrix([x])[0]
# )

In [92]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.with_columns(
#     [
#         pl.col("train").apply(lambda x: to_matrix(list(x))[0]).alias("train"),
#         pl.col("val").apply(lambda x: to_matrix(list('val')[:1])[0]).alias("val"),
#     ]
# )
# df_train_clicks_mapped = df_train_clicks_mapped.sort("user")

In [93]:
# class Dataset_RNN(Dataset):
#     def __init__(self, df, mode):

#         self.df = df  # .reset_index(drop=True)
#         self.mode = mode

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, index):
#         row = self.df.iloc[index]
#         user = row["user"]
#         aids_train = row["train"]
#         # aids_train = to_matrix([aids_train])[0]
#         aids_val = row["val"]
#         # aids_val = to_matrix([aids_val])[0]

#         if self.mode == "test":
#             return torch.tensor(aids_train).long()
#         else:
#             return torch.tensor(aids_train).long(), torch.tensor(aids_val).float()

In [94]:
class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super(Attention, self).__init__()

        self.hidden_size = hidden_size
        self.batch_first = batch_first

        self.att_weights = nn.Parameter(
            torch.Tensor(1, hidden_size), requires_grad=True
        )

        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def get_mask(self):
        pass

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        # print(f"inputs: {inputs.size()}")

        # apply attention layer
        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0)  # (1, hidden_size) --> (hidden_size, 1)
            .unsqueeze(0)  # (1, hidden_size, 1)
            .repeat(batch_size, 1, 1),  # (batch_size, hidden_size, 1)
        )

        # print(f"weights: {weights.size()}")

        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)
        # print(f"attentions: {attentions.size()}")

        # create mask based on the sentence lengths
        mask = torch.ones(attentions.size(), requires_grad=True).to(device)
        # print(f"mask: {mask.size()}")
        if batch_size == 1:
            mask = mask.view(1, -1)
        for i, l in enumerate(lengths):  # skip the first sentence
            if l < max_len:
                mask[i, l:] = 0

        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        _sums = masked.sum(-1).unsqueeze(-1)  # sums per row

        attentions = masked.div(_sums)

        # apply attention weights
        weighted = torch.mul(inputs, attentions.unsqueeze(-1).expand_as(inputs))
        # print(f"weighted: {weighted.size()}")

        # get the final fixed vector representations of the sentences
        representations = weighted.sum(1).squeeze()
        # print(f"representations: {representations.size()}")

        return representations, attentions

In [95]:
# min(600, round(1.6 * n_cat**0.56))

n_vocab = pad + 1
# hidden_dim = 64
# embedding_dim = min(600, round(1.6 * pad**0.56))
# embedding_dim = 64

In [96]:
class RRNN(nn.Module):
    def __init__(
        self,
        n_vocab=n_vocab,
        hidden_dim=config.hidden_dim,
        embedding_dim=config.embedding_dim,
        dropout=0.3,
    ):
        super(RRNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=False)
        self.hidden_dim = hidden_dim
        self.out_features_num = n_vocab
        self.dropout0 = nn.Dropout(dropout, inplace=True)
        self.lstm_desc = nn.GRU(
            embedding_dim,
            hidden_dim,
            dropout=dropout,
            num_layers=1,
            bidirectional=False,
            batch_first=True,
        )

        self.pooling_result_dimension = self.hidden_dim * 1
        self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

        self.attention_descs = Attention(self.hidden_dim * 1, batch_first=True)
        self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
        self.dropout1 = nn.Dropout(dropout, inplace=True)
        self.fc1 = nn.Linear(self.dim_after_concat, self.embedding_dim)
        self.dropout2 = nn.Dropout(dropout, inplace=True)
        self.fc2 = nn.Linear(self.embedding_dim, self.out_features_num)

        self.fc2.weight = self.embeddings.weight

        # self.category_embedding = self._create_embedding_projection(cardinality=10, embed_size=6, padding_idx=None, scale_grad_by_freq=False)
        # self.subcategory_embedding = self._create_embedding_projection(cardinality=50, embed_size=14, padding_idx=None, scale_grad_by_freq=False)
        # self.region_embedding = self._create_embedding_projection(cardinality=85, embed_size=19, padding_idx=None, scale_grad_by_freq=False)

    def _create_embedding_projection(
        self,
        cardinality,
        embed_size,
        add_missing=True,
        padding_idx=0,
        scale_grad_by_freq=False,
    ):
        add_missing = add_missing * 1
        return nn.Embedding(
            num_embeddings=cardinality + add_missing,
            embedding_dim=embed_size,
            padding_idx=padding_idx,
            scale_grad_by_freq=scale_grad_by_freq,
        )

    def forward(self, aids, lens):
        batch_size = aids.shape[0]

        embedded_aids = self.embeddings(aids)
        embedded_aids = self.dropout0(embedded_aids)
        pad_embed_pack = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_aids, list(lens), batch_first=True, enforce_sorted=False
        )
        states_aids, hh_aids = self.lstm_desc(pad_embed_pack)

        states_aids, lens_unpacked = torch.nn.utils.rnn.pad_packed_sequence(
            states_aids, batch_first=True
        )

        lengths_aids = [x_.size(1) for x_ in states_aids]
        atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
        atten_aids = atten_aids.view(batch_size, -1)

        features = torch.cat(
            [
                # subcategory_embed, category_embed, region_embed,
                atten_aids
            ],
            dim=-1,
        )
        features = self.bn1(features)
        features = self.dropout1(features)
        features = features.view(features.size(0), -1)
        features = self.fc1(features)
        features = self.dropout2(features)
        features = self.fc2(features)
        # features = F.normalize(features)

        return features

In [97]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
5108608,"[112608, 117228]",81314
5108608,[112608],117228
6027072,"[154119, 336846, ... 1689]",336846
6027072,[154119],336846
6027072,"[154119, 336846]",193537


In [98]:
# X_val["val"].value_counts()

In [99]:
X_val.head()

user,target,train_seq
i64,list[i64],list[i64]
13340928,"[1713158, 1421960, 1713158]","[213574, 276287, ... 270835]"
13570752,"[1035803, 1775391, 1641216]","[326363, 326363, ... 220643]"
13151872,"[660635, 852900, 1517855]","[339541, 238694]"
13704192,"[188965, 1034864, 188965]","[40288, 370788, ... 86709]"
14419456,"[827115, 313825, 51070]",[243984]


In [100]:
X_train.shape, X_val.shape

((22591260, 3), (466825, 3))

In [101]:
# TRAIN_BATCH_SIZE = 128
# INFER_BATCH_SIZE = 128

# dataset_train = Dataset_RNN(X_train, mode="train")
# dataset_local_valid = Dataset_RNN(X_val, mode="train")

# train_loader = torch.utils.data.DataLoader(
#     dataset_train,
#     batch_size=TRAIN_BATCH_SIZE,
#     shuffle=True,
#     num_workers=0,
#     drop_last=False,
# )
# local_valid_loader = torch.utils.data.DataLoader(
#     dataset_local_valid,
#     batch_size=INFER_BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     drop_last=False,
# )


# dataset_local_inference = Dataset_RNN(X_val, mode='test')
# local_inference_loader = torch.utils.data.DataLoader(
#     dataset_local_inference, batch_size=INFER_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False,
# )

# dataset_inference = Dataset_RNN(df_val, mode='test')
# inference_loader = torch.utils.data.DataLoader(
#     dataset_inference, batch_size=INFER_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False,
# )

In [102]:
# %%time

# X_train["train"] = X_train["train"].progress_apply(lambda x: torch.tensor(x).long())
# X_train["val"] = X_train["val"].progress_apply(lambda x: torch.tensor(x).float())
# X_val["train"] = X_val["train"].progress_apply(lambda x: torch.tensor(x).long())
# X_val["val"] = X_val["val"].progress_apply(lambda x: torch.tensor(x).float())

In [103]:
%%time
X_train = X_train.to_pandas()
X_val = X_val.to_pandas()

X_train_to_dataset = pd.DataFrame(X_train[config.col_train_seq].tolist(), index=X_train.index)
X_train_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_train_to_dataset.columns]
X_train_to_dataset[config.col_target] = X_train[config.col_target]
X_train_to_dataset[config.col_user] = X_train[config.col_user]

X_val_to_dataset = pd.DataFrame(X_val[config.col_train_seq].tolist(), index=X_val.index)
X_val_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_val_to_dataset.columns]
X_val_to_dataset[config.col_target] = X_val[config.col_target]
X_val_to_dataset[config.col_user] = X_val[config.col_user]

X_val_target_to_dataset = pd.DataFrame(X_val_to_dataset[config.col_target].tolist(), index=X_val_to_dataset.index)
X_val_target_to_dataset.columns = ['target' + '_' + str(pos) for pos in X_val_target_to_dataset.columns]
X_val_to_dataset[X_val_target_to_dataset.columns] = X_val_target_to_dataset[X_val_target_to_dataset.columns]
X_val_to_dataset.drop([config.col_target], axis=1, inplace=True)

X_train = pl.from_pandas(X_train)
X_val = pl.from_pandas(X_val)

cols_pos = [col for col in X_train_to_dataset.columns if "pos" in col]
cols_target_val = [col for col in X_val_to_dataset.columns if "target" in col]

CPU times: user 1min 4s, sys: 6.84 s, total: 1min 11s
Wall time: 1min 10s


In [104]:
cols_target_val

['target_0', 'target_1', 'target_2']

In [105]:
X_train_to_dataset.head()

,pos_0,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8,pos_9,target,user
0,112608,117228.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81314,5108608
1,112608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117228,5108608
2,154119,336846.0,193537.0,1689.0,NaN,NaN,NaN,NaN,NaN,NaN,336846,6027072
3,154119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,336846,6027072
4,154119,336846.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193537,6027072


In [106]:
X_val_to_dataset.head()

,pos_0,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8,pos_9,user,target_0,target_1,target_2
0,213574,276287.0,92125.0,63635.0,91314.0,91314.0,203858.0,270835.0,NaN,NaN,13340928,1713158,1421960,1713158
1,326363,326363.0,395762.0,349786.0,349786.0,395762.0,395762.0,339608.0,395762.0,220643.0,13570752,1035803,1775391,1641216
2,339541,238694.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13151872,660635,852900,1517855
3,40288,370788.0,40288.0,40288.0,86709.0,NaN,NaN,NaN,NaN,NaN,13704192,188965,1034864,188965
4,243984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14419456,827115,313825,51070


In [107]:
%%time

X_train_to_dataset.to_parquet('train.parquet', index=False)
X_val_to_dataset.to_parquet('valid.parquet', index=False)

CPU times: user 7.33 s, sys: 720 ms, total: 8.05 s
Wall time: 7.73 s


In [108]:
%%time

train_ds = Dataset_merlin('train.parquet')
train_dl_merlin = Loader(dataset=train_ds, batch_size=config.batch_size_train, shuffle=True)

valid_ds = Dataset_merlin('valid.parquet')
valid_dl_merlin = Loader(dataset=valid_ds, batch_size=config.batch_size_infer, shuffle=False)

CPU times: user 4.6 s, sys: 3.79 s, total: 8.39 s
Wall time: 1.23 s


In [109]:
def get_matrix_and_target_from_batch(
    batch, cols_pos=cols_pos, cols_target_val=cols_target_val, pad=pad, mod="inference"
):
    assert mod in ["inference", "train", "valid"]

    batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
    for i in range(len(cols_pos)):
        batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
    batch_matrix = [
        torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
        for i in range(len(batch_matrix))
    ]
    lens = torch.tensor(list(map(len, batch_matrix)))

    batch_matrix = torch.nn.utils.rnn.pad_sequence(
        batch_matrix, batch_first=True, padding_value=pad  # + 1
    )

    users = batch[config.col_user].squeeze(1).tolist()

    if mod == "train":
        target = batch[config.col_target]
    elif mod == "valid":
        target = torch.zeros([len(batch[cols_target_val[0]]), len(cols_target_val)])
        for i in range(len(cols_target_val)):
            target[:, i] = torch.reshape(batch[cols_target_val[i]], (1, -1))
        target = [
            torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
            for i in range(len(target))
        ]
    elif mod == "inference":
        target = None
    return batch_matrix, target, lens, users

In [110]:
%%time

for batch, _ in tqdm(train_dl_merlin):
    # batch_matrix, batch["val"], lens = get_matrix_and_target_from_batch(batch, cols_pos=cols_pos, pad=pad)
    continue
    # break
# 5 min

  0%|          | 0/22062 [00:00<?, ?it/s]

CPU times: user 10.6 s, sys: 8.74 s, total: 19.3 s
Wall time: 12.4 s


In [111]:
# %%time

# batch_matrix = np.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
# for i in range(len(cols_pos)):
#     batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
# batch_matrix = [
#         torch.tensor([int(val) for val in batch_matrix[i] if not np.isnan(val)])
#         for i in range(len(batch_matrix))
#     ]

In [112]:
# def get_matrix_and_target_from_batch(batch, cols_pos=cols_pos):
#     batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
#     for i in range(len(cols_pos)):
#         batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
#     # batch_target = torch.reshape(batch["val"], (1, -1))
#     return batch_matrix, batch["val"]

In [113]:
DEBUG = False


def train_func(
    train_loader, model, criterion, opts: list, amp=False, num_batches_to_check=1000
):
    model.train()
    bar = tqdm(train_loader)
    losses = list()
    if amp:
        scaler = torch.cuda.amp.GradScaler()

    start_time = time.perf_counter()
    batches_passed = 0
    for batch_idx, (batch, _) in enumerate(bar):
        if num_batches_to_check:
            if batch_idx % num_batches_to_check == 0 and batch_idx > 0:
                batches_passed += num_batches_to_check
                end_time = time.perf_counter()
                total_time = end_time - start_time
                print(f"batches passed: {batches_passed}")
                print(
                    f"last {num_batches_to_check} batches took {total_time / 60 :.4f} minutes"
                )
                print()
                start_time = time.perf_counter()

        aids, targets, lens, users = get_matrix_and_target_from_batch(
            batch, mod="train"
        )
        aids = aids.to(device).long()
        lens = lens.to(device).long()
        # category = category.to(device).long()
        # subcategory = subcategory.to(device).long()
        # region = region.to(device).long()

        targets = targets.to(device).long()
        targets = targets.squeeze(1)

        if DEBUG and batch_idx == 2:
            print("Debug Mode. Only train on first 1 batches.")
            break

        if amp:
            with torch.cuda.amp.autocast():
                logits = model(aids, lens)
                loss = criterion(logits, targets)
                scaler.scale(loss).backward()  # loss.backward()
                for optimizer in opts:
                    scaler.step(optimizer)  # optimizer.step()
                    scaler.update()
        else:
            logits = model(aids, lens)
            # print(logits.size())
            loss = criterion(logits, targets)
            loss.backward()
        for optimizer in opts:
            optimizer.step()

        for optimizer in opts:
            optimizer.zero_grad()

        losses.append(loss.item())
        # smooth_loss = np.mean(losses[-30:])

        # bar.set_description(f"smooth_loss: {smooth_loss:.5f}")

    loss_train = np.mean(losses)
    print("loss_train:", loss_train)
    return loss_train


def do_inverse_transform(array, ordinal_encoder=ordinal_encoder):
    result = ordinal_encoder.inverse_transform(array.reshape(-1, 1)).reshape(1, -1)[0]
    result = [val for val in result if val is not None]
    return result


def flatten_list(l):
    return [item for sublist in l for item in sublist]


def valid_func(valid_loader, model, criterion):
    model.eval()

    bar = tqdm(valid_loader)
    list_preds = list()
    list_targets = list()
    list_users = list()

    losses = list()
    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, targets, lens, users = get_matrix_and_target_from_batch(
                batch, mod="valid"
            )
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            # category = category.to(device).long()
            # subcategory = subcategory.to(device).long()
            # region = region.to(device).long()
            # targets = targets.to(device).long()
            # targets = targets.squeeze(1)

            logits = model(aids, lens)
            # loss = criterion(logits, targets)
            # losses.append(loss.item())

            # PREDS += [torch.argmax(logits, 1).detach().cpu()]
            k = 20
            preds_cur = torch.topk(logits, k).indices.detach().cpu().numpy()
            # print(preds_cur.shape)
            # print(preds_cur)
            # preds_cur = [np.argpartition(-pred, k)[:k] for pred in preds_cur]
            # preds_cur = [
            #     do_inverse_transform(pred, ordinal_encoder=ordinal_encoder)
            #     for pred in preds_cur
            # ]
            list_preds.append(preds_cur)
            targets = [target.tolist() for target in targets]
            list_targets.append(targets)
            list_users.append(users)

            # bar.set_description(f"loss: {loss.item():.5f}")

    # loss_valid = np.mean(losses)
    # print("loss_valid:", loss_valid)

    # PREDS = torch.cat(PREDS).cpu().tolist()
    list_users = flatten_list(list_users)
    list_preds = flatten_list(list_preds)
    list_targets = flatten_list(list_targets)

    # TARGETS = torch.cat(TARGETS).cpu().tolist()
    # print(len(PREDS), len(TARGETS))
    # print(TARGETS)
    # print(PREDS)
    df_val_targets = pd.DataFrame({"session": list_users, "targets": list_targets})

    df_val = pd.DataFrame({"session": list_users, "preds": list_preds})
    df_val = df_val.explode("preds")

    df_val["preds"] = ordinal_encoder.inverse_transform(
        df_val["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_val = df_val.dropna()
    df_val["preds"] = df_val["preds"].astype(int)
    df_val = pl.from_pandas(df_val)
    df_val = (
        (
            df_val.groupby(["session"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("session")
        .to_pandas()
    )
    df_val = df_val_targets.merge(df_val, how="inner", on="session")

    list_preds = df_val["preds"].tolist()
    list_preds = [list(pred) for pred in list_preds]

    list_targets = df_val["targets"].tolist()
    list_targets_top_1 = [target[0] for target in list_targets]

    metric_total = utils.recall_several_targets(list_preds, list_targets)
    print("metric_total:", metric_total)

    metric_top_1 = utils.recall_one_val(list_preds, list_targets_top_1)
    print("metric_top_1:", metric_top_1)

    return df_val


# def inference_func(inference_loader, model):
#     model.eval()

#     bar = tqdm(inference_loader)
#     preds = list()

#     with torch.no_grad():
#         for batch_idx, (descs, category, subcategory, region) in enumerate(bar):
#             descs = descs.to(device).long()
#             category = category.to(device).long()
#             subcategory = subcategory.to(device).long()
#             region = region.to(device).long()

#             logits = model(descs, category, subcategory, region)

#             preds += [logits.detach().cpu()]

#     preds = torch.cat(preds).cpu().numpy()
#     preds = 1.0 / (1.0 + np.exp(-preds))
#     preds = np.concatenate(preds, axis=0)
#     return preds

In [114]:
out_features_num = pad + 1
out_features_num

395763

In [115]:
model = RRNN()
model.to(device)

/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.


/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.




RRNN(
  (embeddings): Embedding(395763, 64)
  (dropout0): Dropout(p=0.3, inplace=True)
  (lstm_desc): GRU(64, 32, batch_first=True, dropout=0.3)
  (attention_descs): Attention()
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=True)
  (fc1): Linear(in_features=32, out_features=64, bias=True)
  (dropout2): Dropout(p=0.3, inplace=True)
  (fc2): Linear(in_features=64, out_features=395763, bias=True)
)

In [116]:
params_sparsed = list()
params_not_sparsed = list()
cnt = 0
for param in model.parameters():
    if cnt == 0:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)
    cnt += 1

In [117]:
params_sparsed[0].shape

torch.Size([395763, 64])

In [118]:
class MultipleOptimizer:
    def __init__(self, *optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for optimizer in self.optimizers:
            optimizer.zero_grad()

    def step(self):
        for optimizer in self.optimizers:
            optimizer.step()


class MultipleScheduler:
    def __init__(self, *schedulers):
        self.schedulers = schedulers

    def step(self):
        for scheduler in self.schedulers:
            scheduler.step()

In [119]:
init_lr = 0.0001

criterion = nn.CrossEntropyLoss(ignore_index=pad)
optimizer = optim.Adam(model.parameters(), lr=init_lr)
# optimizer_sparsed = optim.SparseAdam(params_sparsed, lr=init_lr)
# optimizer_not_sparsed = optim.Adam(params_not_sparsed, lr=init_lr)
opts = [optimizer]  # , optimizer_sparsed, optimizer_not_sparsed
# opt = MultipleOptimizer(optimizer_sparsed, optimizer_not_sparsed)

# scheduler_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler_not_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler = MultipleScheduler(scheduler_sparsed, scheduler_not_sparsed)

In [120]:
%%time

for epoch in tqdm(range(config.n_epochs), desc="epochs"):

    print(f"Epoch {epoch + 1} started")
    torch.cuda.empty_cache()

    # print("LR:", scheduler.schedulers[0].state_dict()["_last_lr"][0])
    loss_train = train_func(train_dl_merlin, model, criterion, opts, num_batches_to_check=None)
    # scheduler.step()

    df_val = valid_func(valid_dl_merlin, model, criterion)
    # X_val['preds'] = preds_local
    # aucs = list()
    # for category, cur_df in X_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('local mean_auc:', np.mean(aucs))

    # loss_valid, preds = valid_func(valid_loader, model, criterion)
    # df_val['preds'] = preds
    # aucs = list()
    # for category, cur_df in df_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('mean_auc:', np.mean(aucs))

    torch.save(
        model.state_dict(),
        f"{config.model_name}_epoch{epoch + 1}.pth",
    )

    print("Checkpoint saved!")
    print()
    print("*" * 90)
    print()
    print()

# del model
torch.cuda.empty_cache()
gc.collect()

epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 started


  0%|          | 0/22062 [00:00<?, ?it/s]

loss_train: 12.69505845667196


  0%|          | 0/456 [00:00<?, ?it/s]

metric_total: 0.09326835537942484
metric_top_1: 0.16854281583034328
Checkpoint saved!

******************************************************************************************


Epoch 2 started


  0%|          | 0/22062 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [147]:
val = X_val_seq[[config.col_user, config.col_train_seq]].clone()
val = val.join(
    pl.from_pandas(df_val), left_on=config.col_user, right_on="session"
).to_pandas()
val["len_seq"] = val[config.col_train_seq].apply(len)
val["target_top1"] = [targets[0] for targets in val["targets"]]
val["targets"] = [targets.tolist() for targets in val["targets"]]
val["preds"] = [preds.tolist() for preds in val["preds"]]

In [148]:
val.head()

,user,train_seq,targets,preds,len_seq,target_top1
0,13340928,"[213574, 276287, 92125, 63635, 91314, 91314, 2...","[1713158, 1421960, 1713158]","[485256, 1551213, 1196256, 615186, 513932, 173...",8,1713158
1,13570752,"[326363, 326363, 395762, 349786, 349786, 39576...","[1035803, 1775391, 1641216]","[1111967, 1126038, 143882, 57315, 1006198, 383...",10,1035803
2,13151872,"[339541, 238694]","[660635, 852900, 1517855]","[332654, 1116864, 1264313, 1624436, 1768724, 3...",2,660635
3,13704192,"[40288, 370788, 40288, 40288, 86709]","[188965, 1034864, 188965]","[188965, 351195, 554660, 1781513, 620813, 8959...",5,188965
4,14419456,[243984],"[827115, 313825, 51070]","[1236775, 558573, 1022566, 1145432, 1294924, 9...",1,827115


In [150]:
%%time

list_len_seq = list()
list_recall_several = list()
list_recall_top1 = list()

for len_seq, df_cur in tqdm(val.groupby('len_seq')):
    list_len_seq.append(len_seq)
    recall_several = utils.recall_several_targets(list(df_cur['preds']), list(df_cur['targets']))
    list_recall_several.append(recall_several)
    recall_top1 = utils.recall_one_val(list(df_cur['preds']), list(df_cur['target_top1']))
    list_recall_top1.append(recall_top1)

  0%|          | 0/10 [00:00<?, ?it/s]

CPU times: user 2.39 s, sys: 4.29 ms, total: 2.39 s
Wall time: 2.38 s


In [151]:
list_len_seq

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [152]:
list_recall_several

[0.10936117443118834,
 0.10211348826480596,
 0.09711932508376384,
 0.09440608584128322,
 0.09073917694083207,
 0.08839149687468853,
 0.08922316093138265,
 0.08614616025107283,
 0.08659266433236484,
 0.07442962574457723]

In [153]:
list_recall_top1

[0.2036373683846966,
 0.19511562193645118,
 0.17095831164150416,
 0.17105887313550835,
 0.15993919945955076,
 0.15612318995344068,
 0.15253451473315474,
 0.15192467815282137,
 0.1502644714296357,
 0.12869478889596522]

In [121]:
# 0.10479944304610936

In [122]:
# metric_total: 0.042083333333333334 1 ep
# metric_total: 0.06625 1 ep 5 iter
# metric_total: 0.059583333333333335 1 ep 10 iter
# metric_total: 0.0725 1 ep 15 iter

In [123]:
X_val[config.col_target].value_counts().head(10)

target,counts
list[i64],u32
"[1517692, 957481, 957481]",1
"[258353, 50933, 1094372]",1
"[6851, 342019, 776978]",1
"[1393958, 230827, 1570902]",1
"[1689819, 1489275, 1489275]",1
"[352756, 1679224, 352756]",1
"[29735, 832192, 861401]",1
"[1325167, 743473, 921598]",1
"[436838, 436838, 234997]",2


In [124]:
X_val = X_val.to_pandas()

In [125]:
X_val.shape

(466825, 3)

In [126]:
X_val.tail()

,user,target,train_seq
466820,13147201,"[1357776, 1355234, 1357776]",[231368]
466821,12957313,"[268511, 84824, 84824]","[104041, 104041, 104041, 381647, 104041, 38164..."
466822,13235393,"[1695122, 796009, 796009]","[169437, 169437]"
466823,12902593,"[1146494, 1146494, 484201]","[395762, 202546, 246749, 118491]"
466824,13054081,"[384376, 278354, 997384]",[395762]


In [127]:
stop

NameError: name 'stop' is not defined

In [ ]:
%%time

PATH_DATA = '../data/02_intermediate/'
# df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
# df_train = df_train.to_pandas()
df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

# df_train.name = "df_train"
df_test.name = "df_test"

# df_train = utils.make_date_stats(df_train)
df_test = utils.make_date_stats(df_test)

# df_train.drop(['date_day'], axis=1, inplace=True)
df_test.drop(['date_day'], axis=1, inplace=True)

# df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

In [ ]:
df_test = df_test.sort(config.col_ts)

In [ ]:
df_test = df_test.to_pandas()
df_test[config.col_aid] = ordinal_encoder.transform(
    df_test[config.col_aid].values.reshape(-1, 1)
).astype(int)
df_test = pl.from_pandas(df_test)

In [ ]:
%%time

df_test = (
    df_test.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list().alias(config.col_train_seq),
        ]
    )
)

In [ ]:
%%time

df_test = df_test.to_pandas()

to_dataset = pd.DataFrame(df_test[config.col_train_seq].tolist(), index=df_test.index)
to_dataset.columns = ['pos' + '_' + str(pos) for pos in to_dataset.columns]
to_dataset[config.col_user] = df_test[config.col_user]

df_test = pl.from_pandas(df_test)

cols_pos = [col for col in to_dataset.columns if "pos" in col]

PATH_DATASET = 'to_dataset.parquet'

to_dataset.to_parquet(PATH_DATASET, index=False)

dataset = Dataset_merlin(PATH_DATASET)
dl_merlin = Loader(dataset=dataset, batch_size=config.batch_size_infer, shuffle=False)

In [ ]:
def inference_func(loader, model, num_candidates=20):
    model.eval()

    bar = tqdm(loader)
    list_preds = list()
    list_users = list()

    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, _, lens, users = get_matrix_and_target_from_batch(
                batch, mod="inference"
            )
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            logits = model(aids, lens)
            preds_cur = (
                torch.topk(logits, num_candidates).indices.detach().cpu().numpy()
            )
            list_preds.append(preds_cur)
            list_users.append(users)

    list_preds = utils.flatten_list(list_preds)
    list_users = utils.flatten_list(list_users)

    df_preds = pd.DataFrame({"session": list_users, "preds": list_preds})
    df_preds = df_preds.explode("preds")

    df_preds["preds"] = ordinal_encoder.inverse_transform(
        df_preds["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_preds = df_preds.dropna()
    df_preds["preds"] = df_preds["preds"].astype(int)
    df_preds = pl.from_pandas(df_preds)
    df_preds = (
        (
            df_preds.groupby(["session"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("session")
        .to_pandas()
    )

    # PREDS = df_preds["preds"].tolist()
    # PREDS = [list(pred) for pred in PREDS]

    return df_preds

In [ ]:
sub = pd.read_csv("../data/01_raw/sample_submission.csv")

In [ ]:
%%time

model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{0 + 1}.pth",
    )
)

model.to(device)

df_preds = inference_func(dl_merlin, model)
df_preds["session"] = list(map(str, df_preds["session"]))

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

pred_df.to_csv(f"../data/07_model_output/sub_last_week_ep1.csv", index=False)

In [ ]:
%%time

model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{1 + 1}.pth",
    )
)

model.to(device)

df_preds = inference_func(dl_merlin, model)
df_preds["session"] = list(map(str, df_preds["session"]))

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

pred_df.to_csv(f"../data/07_model_output/sub_last_week_ep2.csv", index=False)

In [ ]:
%%time

model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{4 + 1}.pth",
    )
)

model.to(device)

df_preds = inference_func(dl_merlin, model)
df_preds["session"] = list(map(str, df_preds["session"]))

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

pred_df.to_csv(f"../data/07_model_output/sub_last_week_ep5.csv", index=False)

In [ ]:
stop

In [ ]:
X_val[X_val["recall_row"] == 1].shape

In [ ]:
X_val_true = X_val[X_val["recall_row"] == 1]

In [ ]:
pad

In [ ]:
X_val_true.head()

In [ ]:
X_train[X_train["val"] == pad].shape[0] / X_train.shape[0]

In [ ]:
X_val_true.iloc[0]["preds"]

In [ ]:
stop
# loss_train: 12.190348625183105

# loss_train: 9.231197555859884
# loss_valid: 20.559587955474854
# metric_total: 0.000999000999000999

In [ ]:
params_sparsed = list()
params_not_sparsed = list()
for param in model.parameters():
    if param.grad.is_sparse:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)

In [ ]:
params_sparsed

In [ ]:
df_train.groupby(["n_week"]).count().sort("n_week", reverse=False)

In [ ]:
df_train_ex = df_train.filter(pl.col("user") <= 10000)

In [ ]:
%%time

dict_train = dict()
dict_answers = dict()
dict_val_len = dict()
dict_user = dict()
key = 1

for df_user in tqdm(df_train.groupby(["user"], maintain_order=True)):
    df_weeks_count = df_user.groupby(["n_week"]).count().sort("n_week", reverse=False)
    if len(df_weeks_count) <= 1:
        continue
    user = df_user["user"][0]
    for i in range(len(df_weeks_count["n_week"]) - 1):
        n_week_cur = df_weeks_count["n_week"][i]
        val_len = 34 - n_week_cur
        dict_train[key] = df_user.filter(pl.col("n_week") == n_week_cur)
        dict_answers[key] = df_user.filter(pl.col("n_week") > n_week_next)
        dict_val_len[key] = val_len
        dict_user[key] = user
        key += 1

In [ ]:
dict_train[1]

In [ ]:
dict_answers[1]

In [ ]:
df_user.filter(pl.col("n_week") == n_week_cur)

In [ ]:
for i in range(len(df_weeks_count["n_week"]) - 1):
    n_week_cur = df_weeks_count["n_week"][i]
    n_week_next = df_weeks_count["n_week"][i + 1]
    print(n_week_cur, n_week_next)

In [ ]:
df_user

In [ ]:
df_train_ex

In [ ]:
df_train.tail(10)

In [ ]:
df_test.head()

In [ ]:
utils.plot_user_activity(user=12899774, df=df_train)

In [ ]:
df_train["type"].value_counts()

In [ ]:
df_train["date"].min(), df_train["date"].max()

In [ ]:
df_test["date"].min(), df_test["date"].max()

In [ ]:
%%time

df_train.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
%%time

df_test.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
df_train_n_session = (df_train.groupby("user")[["n_session"]].max() + 1).reset_index()
df_test_n_session = (df_test.groupby("user")[["n_session"]].max() + 1).reset_index()

In [ ]:
df_train_n_session["n_session"].value_counts()

In [ ]:
df_test_n_session["n_session"].value_counts()

In [ ]:
def plot_n_interactions_per_dayofweek(df):
    try:
        df_name = df.name
    except AttributeError:
        df_name = None

    title = "n_interactions per dayofweek"
    if df_name is not None:
        title += f" in {df_name}"

    plt.figure(figsize=(12, 7))
    plt.title(title)
    plt.xlabel("dayofweek")
    plt.ylabel("n_interactions")

    dict_type_inter_offset = dict()
    dict_type_inter_offset["clicks"] = -0.2
    dict_type_inter_offset["carts"] = 0.0
    dict_type_inter_offset["orders"] = 0.2

    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "g"
    dict_type_inter_color["orders"] = "r"

    for (dayofweek, type_inter), df_cur in df.groupby(["dayofweek", "type"]):
        ys = df_cur["ts"].count()
        plt.bar(
            dayofweek + dict_type_inter_offset[type_inter],
            ys,
            width=0.2,
            color=dict_type_inter_color[type_inter],
            align="center",
            # label=feature,
            # marker=",",
            # linestyle="dashed",
            alpha=0.6,
        )

In [ ]:
plot_n_interactions_per_dayofweek(df_train)

In [ ]:
plot_n_interactions_per_dayofweek(df_test)

In [ ]:
def plot_user_activity(user, df):
    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "orange"
    dict_type_inter_color["orders"] = "r"

    offset = 0.5
    df_cur = df[df["user"] == user]
    df_cur["date_day"] = df_cur["date_day"].astype(str)

    if df.name == "df_train":
        x = pd.date_range("2022-08-01", "2022-08-28")
    elif df.name == "df_test":
        x = pd.date_range("2022-08-29", "2022-09-04")
    x = [str(d).split(" ")[0] for d in x]
    map_dates = {x[i]: i for i in range(len(x))}
    points_sundays = [point for point in range(len(x)) if (point + 1) % 7 == 0]

    plt.figure(figsize=(12, 7))
    plt.xticks(range(len(x)), x, rotation=90)
    plt.yticks(range(25))

    for point in range(25):
        plt.axhline(y=point, color="gray", linestyle="-", alpha=0.1)
    for point in range(len(x)):
        plt.axvline(x=point + offset, color="gray", linestyle="-", alpha=0.1)
        plt.axvline(x=point - offset, color="gray", linestyle="-", alpha=0.1)
    for point in points_sundays:
        plt.axvline(x=point + offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset - 1, color="r", linestyle="-", alpha=0.3)

    for action, df_cur_action in df_cur.groupby("type"):
        xx = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        yy = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        plt.scatter(
            df_cur_action["date_day"].map(map_dates) + xx,
            df_cur_action["hour"] + yy,
            c=dict_type_inter_color[action],
            s=10,
            label=action,
        )
    plt.legend(
        loc="upper right", bbox_to_anchor=(1.1, 1.0), ncol=1, fancybox=True, shadow=True
    )

In [ ]:
plot_user_activity(user=2932269, df=df_train)

In [ ]:
plot_user_activity(user=12899780, df=df_test)